In [1]:
#스레드 사용법
# threading 모듈 추
import time, threading

def send(n):
    for i in range(n):
        print("send msg:", i)
        time.sleep(1)

def recv(n):
    for i in range(n):
        print("recv msg:", i)
        time.sleep(1)

if __name__ == "__main__":  # __name__ == "__main__"아래 코드는 직접 이 파일을 실행할 때만 동작함
    # 스레드 객체 생성하고 실행하기
    sender = threading.Thread(target=send, args=(5,))
    sender.start()
    receiver = threading.Thread(target=recv, args=(5,))
    receiver.start()

send msg: 0
recv msg: 0
send msg: 1
recv msg: 1
send msg: 2
recv msg: 2
send msg: 3
recv msg: 3
recv msg:send msg: 4
 4


In [2]:
# 스레드 생명주기

import threading, time

count = 0

def worker(n):
    global count
    print("worker thread start")
    for i in range(n):
        count+=1
        time.sleep(1)
    print("worker thread end")

if __name__ == "__main__":
    # 스레드 객체 생성 및 실행
    t = threading.Thread(target=worker, args=(5,))
    t.start()
    # 서브 스레드가 종료될때까지 메인 스레드가 대기하려면 join()사용
    t.join()
    
    # count 출력
    print("count:",count)

worker thread start
worker thread end
count: 5


In [ ]:
# # 무한루프 스레드 종료하기
# # 방법 1: daemon thread로 만들기 (메인 종료시 서브스레드는 같이 종료) -> t.daemon = True

# import threading, time

# event = threading.Event()

# def loop():
#     # while True:
#     #     time.sleep(1)
#     #     print("Infinite Loop thread!")

# if __name__=="__main__":
#     # 스레드 객체생성 후 실행
#     t = threading.Thread(target=loop)
#     t.start()
#     t.daemon = True
    
#     print("main thread start")
#     time.sleep(5)
    
#     print("main thread end")

In [5]:
# 무한루프 스레드 종료하기
# 방법 2: 스레드 이벤트 객체 사용
# Event 객체의 set()을 호출하면 객체의 내부 플래그 값을 True로 설정
# is_set()을 사용해 플래그 값을 확인 후 서브 스레드 종료

import threading, time

#이벤트 객체 생성
event = threading.Event()

def loop():
    while True:
        time.sleep(1)
        if event.is_set(): # 이벤트 객체 flag 확인
            print("Infinite Loop thread stop!")
            return
        print("Infinite Loop thread!")

if __name__=="__main__":
    # 스레드 객체생성 후 실행
    t = threading.Thread(target=loop)
    t.start()

    print("main thread start")
    time.sleep(5)
    event.set() # 이벤트 객체 flag set
    print("main thread end")

main thread start
Infinite Loop thread!
Infinite Loop thread!
Infinite Loop thread!
Infinite Loop thread!
Infinite Loop thread!
Infinite Loop thread!


 1


main thread end
Infinite Loop thread stop!


In [6]:
# 스레드간 통신하기

import threading

def total_sum(start,end):
    result = 0
    for i in range(start, end):
        result+=i
    print("result:",result)

if __name__ == "__main__":
    start = 0
    end = 1000000

    t = threading.Thread(target=total_sum, args=(start,end+1))
    t.start()

result: 500000500000


In [19]:
# 스레드간 통신하기
# 방법1: list 사용하기

import threading

def total_sum(start,end, sum_list):
    result = 0
    for i in range(start, end):
        result+=i
    print("result:",result)
    sum_list.append(result)
    return

if __name__ == "__main__":
    numbers = list()
    start = 0
    end = 1000000

    t1 = threading.Thread(target=total_sum, args=(start,end//2,numbers))
    t2 = threading.Thread(target=total_sum, args=(end//2,end+1,numbers))
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    
    print("total sum:",sum(numbers))

result: 124999750000
result: 375000750000
total sum: 500000500000


In [18]:
# 스레드간 통신하기
# 방법2: Que 사용하기 ( 일반적인 방법 )

import threading, queue

def total_sum(start,end, q):
    result = 0
    for i in range(start, end):
        result+=i
    q.put(result)
    return

if __name__ == "__main__":
    q = queue.Queue() # 큐 생성
    
    start = 0
    end = 1000000

    t1 = threading.Thread(target=total_sum, args=(start,end//2,q))
    t2 = threading.Thread(target=total_sum, args=(end//2,end+1,q))
    t1.start()
    t2.start()
    t1.join()
    t2.join()

    q.put("Done")
    result = 0

    while True:
        data = q.get()
        if data == "Done":
            break
        else:
            result += data
    
    print("total sum:",result)

total sum: 500000500000


In [27]:
# 스레드 레이스
import threading, time, queue

def turtle(winner):
    for i in range(1,31):
        time.sleep(1)
        print(f"거북이-{i}m")
    print("거북이-30m 도착")
    winner.put('turtle')
    return

# 토끼는 0.5초에 1m의 속도로 달림
# 25m 지점에서 낮잠 (sleep 사용)
# 낮잠에서 깨어나면 나머지 구간을 0.3초에 1m 속도로 달림
def rabbit(winner):
    for i in range(1,31):
        if i < 25:
            time.sleep(0.5)
            print(f"토끼-{i}m")
        elif i == 25:
            time.sleep(0.5)
            print(f"토끼-{i}m 낮잠")
            time.sleep(17)
            print(f"토끼-{i}m 잠깸")
        else:
            time.sleep(0.3)
            print(f"토끼-{i}m")
    print("토끼-30m 도착")
    winner.put('rabbit')
    return

if __name__=="__main__":
    winner = queue.Queue()
    print("토끼와 거북이 경주 시작")
    t1 = threading.Thread(target=rabbit,args=(winner,))
    t1.start()
    t2 = threading.Thread(target=turtle,args=(winner,))
    t2.start()
    t1.join()
    t2.join()
    
    print("토끼와 거북이 경주의 우승자: ",winner.get())
    
    

토끼와 거북이 경주 시작
토끼-1m
거북이-1m
토끼-2m
토끼-3m
거북이-2m
토끼-4m
토끼-5m
거북이-3m
토끼-6m
토끼-7m
거북이-4m
토끼-8m
토끼-9m
거북이-5m
토끼-10m
토끼-11m
거북이-6m
토끼-12m
토끼-13m
거북이-7m
토끼-14m
토끼-15m
거북이-8m
토끼-16m
토끼-17m
거북이-9m
토끼-18m
토끼-19m
거북이-10m
토끼-20m
토끼-21m
거북이-11m
토끼-22m
토끼-23m
거북이-12m
토끼-24m
토끼-25m 낮잠
거북이-13m
거북이-14m
거북이-15m
거북이-16m
거북이-17m
거북이-18m
거북이-19m
거북이-20m
거북이-21m
거북이-22m
거북이-23m
거북이-24m
거북이-25m
거북이-26m
거북이-27m
거북이-28m
거북이-29m
토끼-25m 잠깸
토끼-26m
거북이-30m
거북이-30m 도착
토끼-27m
토끼-28m
토끼-29m
토끼-30m
우승자:  turtle
